In [ ]:
# Setup: clone the repo and switch to the `cmkd` branch (all CMKD code lives there, not on main)
!git clone https://github.com/TotallyNotMinh/Environmental-Sound-Recognition-Under-Unseen-Conditions.git
%cd /kaggle/working/Environmental-Sound-Recognition-Under-Unseen-Conditions
!git fetch origin
!git checkout cmkd
!git pull origin cmkd

In [ ]:
!pip install -q h5py scipy opencv-python tqdm

In [ ]:
# Quick check (~2 min, no dataset needed): CNN training + evaluation on synthetic data.
!python scripts/train_cnn_classifier.py --mock --num-epoch 2 --num-workers 0 --checkpoint-dir /kaggle/working/smoke/cnn/
!python scripts/evaluate.py --mock --checkpoint /kaggle/working/smoke/cnn/best_cnn.pth --num-workers 0

In [ ]:
# Train the CNN baseline (EfficientNet-B0, no KD). CMKD FSD50K CNN recipe: LR 5e-4, batch 24, 50 epochs, BCE.
# --batch-size is the TOTAL batch; with 2 GPUs each GPU gets 12.
#
# Kaggle stops a session at 12 h, so training is capped at 11 h (timeout) to leave time for evaluation
# and for Kaggle to save /kaggle/working as this version's output.
# To CONTINUE an unfinished run: "Add Input" -> this notebook's previous version output, then run again.
# The cell below finds that checkpoint, copies it here, and training resumes from the last finished epoch.
import glob, os, shutil

RUN = "/kaggle/working/checkpoints/fsd50k_cnn_b0"
os.makedirs(RUN, exist_ok=True)
prev = [p for d in glob.glob("/kaggle/input/*") if not d.endswith("datasets")
        for p in glob.glob(f"{d}/**/fsd50k_cnn_b0/checkpoint.pth", recursive=True)]
if prev and not os.path.exists(f"{RUN}/checkpoint.pth"):
    src = os.path.dirname(prev[0])
    for f in os.listdir(src):
        shutil.copy(os.path.join(src, f), RUN)
    print(f"Resuming from previous run: {src}")
else:
    print("No previous checkpoint found -- starting fresh." if not prev else "Checkpoint already in /kaggle/working -- resuming.")

!timeout --signal=INT --kill-after=5m 11h torchrun --nproc_per_node=2 scripts/train_cnn_classifier.py \
        --data-path /kaggle/input/datasets/yousirui1/fsd50k/fsd50k \
        --batch-size 24 \
        --lr 5e-4 \
        --num-epoch 50 \
        --checkpoint-dir {RUN}/ \
        --checkpoint-path {RUN}/checkpoint.pth

In [ ]:
# CNN performance: score the best checkpoint on the FSD50K EVAL split (the split the CMKD paper reports).
# Results are printed and saved to {RUN}/eval_eval_best_cnn.json -- compare this with the AST / KD-AST runs.
import pandas as pd

hist = pd.read_csv(f"{RUN}/metrics.csv")
last = int(hist["epoch"].max())
print(f"Epochs finished: {last}/50" + ("" if last >= 50 else "  <-- unfinished: add this version's output as input and re-run to continue"))
print(hist.tail(5).to_string(index=False))

!python scripts/evaluate.py \
        --checkpoint {RUN}/best_cnn.pth \
        --data-path /kaggle/input/datasets/yousirui1/fsd50k/fsd50k \
        --split eval